In [1]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path

spark = create_spark_session(
    "NYC Building Risk - Building Identity Test"
)

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/05 15:13:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.4.0
Spark UI: http://ff7529cf1dd6:4040


In [2]:
pluto_df = spark.read.parquet(
    minio_path("silver/pluto/version=26v2")
)

nyc311_df = spark.read.parquet(
    minio_path("silver/nyc_311")
)

hpd_df = spark.read.parquet(
    minio_path("silver/hpd")
)

dob_df = spark.read.parquet(
    minio_path("silver/dob")
)

print("PLUTO:", pluto_df.count())
print("311:", nyc311_df.count())
print("HPD:", hpd_df.count())
print("DOB:", dob_df.count())

26/09/05 15:16:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


PLUTO: 858284


311: 885306


HPD: 927308
DOB: 148688


In [5]:
from pyspark.sql import functions as F


# PLUTO = master list of valid building / tax-lot BBLs
pluto_bbl_df = (
    pluto_df
    .select("bbl")
    .filter(F.col("bbl").isNotNull())
    .dropDuplicates()
)


def check_join_coverage(source_df, source_name):

    total_rows = source_df.count()

    missing_bbl = (
        source_df
        .filter(F.col("bbl").isNull())
        .count()
    )

    matched_rows = (
        source_df
        .select("bbl")
        .join(
            pluto_bbl_df,
            on="bbl",
            how="left_semi"
        )
        .count()
    )

    unmatched_rows = (
        total_rows - matched_rows
    )

    match_percent = (
        matched_rows / total_rows * 100
        if total_rows > 0
        else 0
    )

    print()
    print("========================================")
    print(source_name)
    print("========================================")

    print(f"Total rows:      {total_rows:,}")
    print(f"Matched PLUTO:   {matched_rows:,}")
    print(f"Unmatched:       {unmatched_rows:,}")
    print(f"Missing BBL:     {missing_bbl:,}")
    print(f"Match coverage:  {match_percent:.2f}%")

In [6]:
check_join_coverage(
    nyc311_df,
    "NYC 311"
)

check_join_coverage(
    hpd_df,
    "HPD"
)

check_join_coverage(
    dob_df,
    "DOB"
)


NYC 311
Total rows:      885,306
Matched PLUTO:   880,463
Unmatched:       4,843
Missing BBL:     2,667
Match coverage:  99.45%



HPD
Total rows:      927,308
Matched PLUTO:   926,159
Unmatched:       1,149
Missing BBL:     0
Match coverage:  99.88%

DOB
Total rows:      148,688
Matched PLUTO:   147,422
Unmatched:       1,266
Missing BBL:     137
Match coverage:  99.15%


In [7]:
def get_unmatched_bbls(source_df):

    return (
        source_df
        .filter(
            F.col("bbl").isNotNull()
        )
        .select("bbl")
        .join(
            pluto_bbl_df,
            on="bbl",
            how="left_anti"
        )
        .groupBy("bbl")
        .count()
        .orderBy(
            F.desc("count")
        )
    )


unmatched_311_bbl = get_unmatched_bbls(
    nyc311_df
)

unmatched_hpd_bbl = get_unmatched_bbls(
    hpd_df
)

unmatched_dob_bbl = get_unmatched_bbls(
    dob_df
)

In [8]:
print(
    "311 unmatched unique BBL:",
    unmatched_311_bbl.count()
)

print(
    "HPD unmatched unique BBL:",
    unmatched_hpd_bbl.count()
)

print(
    "DOB unmatched unique BBL:",
    unmatched_dob_bbl.count()
)

311 unmatched unique BBL: 137


HPD unmatched unique BBL: 289
DOB unmatched unique BBL: 464


In [9]:
print("NYC 311:")
unmatched_311_bbl.show(
    20,
    truncate=False
)

print("HPD:")
unmatched_hpd_bbl.show(
    20,
    truncate=False
)

print("DOB:")
unmatched_dob_bbl.show(
    20,
    truncate=False
)

NYC 311:


+----------+-----+
|bbl       |count|
+----------+-----+
|2024439080|338  |
|1020019005|246  |
|3020030037|246  |
|3044960015|92   |
|2028500063|79   |
|2030880039|56   |
|4033609001|49   |
|1019290158|47   |
|3038610001|44   |
|3051050008|42   |
|2028720148|38   |
|3033480010|38   |
|3024720410|37   |
|3045860201|36   |
|3016990033|33   |
|1021750001|30   |
|3051370025|30   |
|4157050069|29   |
|3018689028|29   |
|1018420055|27   |
+----------+-----+
only showing top 20 rows

HPD:


+----------+-----+
|bbl       |count|
+----------+-----+
|2024439080|111  |
|1020019005|58   |
|3018689028|37   |
|3020030037|32   |
|3044960015|31   |
|1018420053|29   |
|3033480010|26   |
|2028500063|25   |
|1019290158|25   |
|4033609001|23   |
|3051370025|19   |
|2031050055|18   |
|2030880039|18   |
|4021050001|16   |
|1018420055|16   |
|2023690010|14   |
|4157050069|12   |
|3017780055|11   |
|3038610001|10   |
|3016990033|10   |
+----------+-----+
only showing top 20 rows

DOB:


+----------+-----+
|bbl       |count|
+----------+-----+
|1012809010|76   |
|1007297503|19   |
|3001490100|15   |
|1008070050|12   |
|3037660001|12   |
|3024140025|12   |
|1013030053|11   |
|1008120042|11   |
|1011710164|11   |
|4051020011|10   |
|1003150045|10   |
|1000920030|10   |
|2024209078|10   |
|3055170065|10   |
|3058730077|10   |
|1006390001|9    |
|1000280005|8    |
|1007350035|8    |
|2024209020|8    |
|1005540001|8    |
+----------+-----+
only showing top 20 rows



In [10]:
u311 = unmatched_311_bbl.select("bbl")
uhpd = unmatched_hpd_bbl.select("bbl")
udob = unmatched_dob_bbl.select("bbl")


shared_311_hpd = (
    u311
    .join(
        uhpd,
        on="bbl",
        how="inner"
    )
)

shared_311_dob = (
    u311
    .join(
        udob,
        on="bbl",
        how="inner"
    )
)

shared_hpd_dob = (
    uhpd
    .join(
        udob,
        on="bbl",
        how="inner"
    )
)

shared_all_three = (
    u311
    .join(
        uhpd,
        on="bbl",
        how="inner"
    )
    .join(
        udob,
        on="bbl",
        how="inner"
    )
)


print(
    "Shared 311 + HPD:",
    shared_311_hpd.count()
)

print(
    "Shared 311 + DOB:",
    shared_311_dob.count()
)

print(
    "Shared HPD + DOB:",
    shared_hpd_dob.count()
)

print(
    "Shared all three:",
    shared_all_three.count()
)

Shared 311 + HPD: 87


Shared 311 + DOB: 39


Shared HPD + DOB: 52


Shared all three: 25


In [11]:
shared_311_hpd_top = (
    unmatched_311_bbl
    .withColumnRenamed(
        "count",
        "count_311"
    )
    .join(
        unmatched_hpd_bbl
        .withColumnRenamed(
            "count",
            "count_hpd"
        ),
        on="bbl",
        how="inner"
    )
    .withColumn(
        "total_events",
        F.col("count_311")
        + F.col("count_hpd")
    )
    .orderBy(
        F.desc("total_events")
    )
)

shared_311_hpd_top.show(
    30,
    truncate=False
)

+----------+---------+---------+------------+
|bbl       |count_311|count_hpd|total_events|
+----------+---------+---------+------------+
|2024439080|338      |111      |449         |
|1020019005|246      |58       |304         |
|3020030037|246      |32       |278         |
|3044960015|92       |31       |123         |
|2028500063|79       |25       |104         |
|2030880039|56       |18       |74          |
|4033609001|49       |23       |72          |
|1019290158|47       |25       |72          |
|3018689028|29       |37       |66          |
|3033480010|38       |26       |64          |
|3038610001|44       |10       |54          |
|3051370025|30       |19       |49          |
|3051050008|42       |4        |46          |
|2028720148|38       |7        |45          |
|3045860201|36       |8        |44          |
|3024720410|37       |6        |43          |
|3016990033|33       |10       |43          |
|1018420055|27       |16       |43          |
|4157050069|29       |12       |41

In [12]:
test_bbl = "2024439080"


print("========== NYC 311 ==========")

nyc311_df.filter(
    F.col("bbl") == test_bbl
).select(
    "bbl",
    "incident_address",
    "borough",
    "latitude",
    "longitude",
    "complaint_type"
).show(
    20,
    truncate=False
)


print("========== HPD ==========")

hpd_df.filter(
    F.col("bbl") == test_bbl
).select(
    "bbl",
    "housenumber",
    "streetname",
    "boro",
    "bin",
    "latitude",
    "longitude",
    "class",
    "violationstatus"
).show(
    20,
    truncate=False
)


print("========== DOB ==========")

dob_df.filter(
    F.col("bbl") == test_bbl
).select(
    "bbl",
    "house_number",
    "street",
    "borough",
    "bin",
    "latitude",
    "longitude",
    "violation_type",
    "violation_status"
).show(
    20,
    truncate=False
)

========== NYC 311 ==========
+----------+--------------------------+-------+------------------+------------------+--------------+
|bbl       |incident_address          |borough|latitude          |longitude         |complaint_type|
+----------+--------------------------+-------+------------------+------------------+--------------+
|2024439080|800 CONCOURSE VILLAGE WEST|BRONX  |40.82478464305234 |-73.92237367726554|HEAT/HOT WATER|
|2024439080|800 CONCOURSE VILLAGE WEST|BRONX  |40.82478464305234 |-73.92237367726554|HEAT/HOT WATER|
|2024439080|780 CONCOURSE VILLAGE WEST|BRONX  |40.82392039964646 |-73.9228841423698 |HEAT/HOT WATER|
|2024439080|800 CONCOURSE VILLAGE WEST|BRONX  |40.82478464305234 |-73.92237367726554|HEAT/HOT WATER|
|2024439080|780 CONCOURSE VILLAGE WEST|BRONX  |40.82392039964646 |-73.9228841423698 |HEAT/HOT WATER|
|2024439080|790 CONCOURSE VILLAGE WEST|BRONX  |40.824342918781106|-73.9226343424295 |HEAT/HOT WATER|
|2024439080|800 CONCOURSE VILLAGE WEST|BRONX  |40.82478464305

+----------+-----------+----------------------+-----+-------+---------+----------+-----+---------------+
|bbl       |housenumber|streetname            |boro |bin    |latitude |longitude |class|violationstatus|
+----------+-----------+----------------------+-----+-------+---------+----------+-----+---------------+
|2024439080|780        |CONCOURSE VILLAGE WEST|BRONX|2002455|40.823921|-73.922884|B    |OPEN           |
|2024439080|780        |CONCOURSE VILLAGE WEST|BRONX|2002455|40.823921|-73.922884|B    |OPEN           |
|2024439080|780        |CONCOURSE VILLAGE WEST|BRONX|2002455|40.823921|-73.922884|B    |OPEN           |
|2024439080|800        |CONCOURSE VILLAGE WEST|BRONX|2101572|40.824785|-73.922374|A    |CLOSE          |
|2024439080|790        |CONCOURSE VILLAGE WEST|BRONX|2101571|40.824343|-73.922634|B    |CLOSE          |
|2024439080|800        |CONCOURSE VILLAGE WEST|BRONX|2101572|40.824785|-73.922374|C    |OPEN           |
|2024439080|790        |CONCOURSE VILLAGE WEST|BRONX|21

In [13]:
from pyspark.sql import functions as F


# ==================================================
# 1. HPD -> COMMON BUILDING STRUCTURE
# ==================================================

hpd_buildings = (
    hpd_df

    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )

    .select(
        F.trim(F.col("bin")).alias("bin"),
        F.col("bbl"),

        F.concat_ws(
            " ",
            F.trim(F.col("housenumber")),
            F.trim(F.col("streetname"))
        ).alias("address"),

        F.col("boro").alias("borough"),
        F.col("latitude"),
        F.col("longitude"),

        F.lit("HPD").alias("source")
    )
)


# ==================================================
# 2. DOB -> COMMON BUILDING STRUCTURE
# ==================================================

dob_buildings = (
    dob_df

    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )

    .select(
        F.trim(F.col("bin")).alias("bin"),
        F.col("bbl"),

        F.concat_ws(
            " ",
            F.trim(F.col("house_number")),
            F.trim(F.col("street"))
        ).alias("address"),

        F.col("borough"),
        F.col("latitude"),
        F.col("longitude"),

        F.lit("DOB").alias("source")
    )
)


# ==================================================
# 3. COMBINE HPD + DOB
# ==================================================

building_candidates = (
    hpd_buildings
    .unionByName(dob_buildings)
)


print(
    "Building candidate rows:",
    building_candidates.count()
)

print(
    "Unique BIN:",
    building_candidates
    .select("bin")
    .distinct()
    .count()
)

Building candidate rows: 1075193


Unique BIN: 197958


In [14]:
bin_quality = (
    building_candidates

    .groupBy("bin")

    .agg(
        F.count("*").alias(
            "records"
        ),

        F.countDistinct(
            "bbl"
        ).alias(
            "distinct_bbl"
        ),

        F.countDistinct(
            "address"
        ).alias(
            "distinct_address"
        ),

        F.countDistinct(
            "borough"
        ).alias(
            "distinct_borough"
        ),

        F.countDistinct(
            "source"
        ).alias(
            "source_count"
        )
    )
)


print(
    "BIN with more than one BBL:",
    bin_quality
    .filter(
        F.col("distinct_bbl") > 1
    )
    .count()
)

print(
    "BIN with more than one address:",
    bin_quality
    .filter(
        F.col("distinct_address") > 1
    )
    .count()
)

print(
    "BIN appearing in both HPD and DOB:",
    bin_quality
    .filter(
        F.col("source_count") == 2
    )
    .count()
)

BIN with more than one BBL: 167


BIN with more than one address: 5610


BIN appearing in both HPD and DOB: 33888


In [15]:
from pyspark.sql import functions as F


hpd_buildings = (
    hpd_df
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
    .select(
        F.trim(F.col("bin")).alias("bin"),
        F.col("bbl"),

        F.upper(
            F.trim(
                F.concat_ws(
                    " ",
                    F.col("housenumber"),
                    F.col("streetname")
                )
            )
        ).alias("address"),

        F.col("boro").alias("borough"),
        F.col("latitude"),
        F.col("longitude"),

        F.col("inspectiondate").alias("event_date"),

        F.lit("HPD").alias("source")
    )
)


dob_buildings = (
    dob_df
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
    .select(
        F.trim(F.col("bin")).alias("bin"),
        F.col("bbl"),

        F.upper(
            F.trim(
                F.concat_ws(
                    " ",
                    F.col("house_number"),
                    F.col("street")
                )
            )
        ).alias("address"),

        F.col("borough"),
        F.col("latitude"),
        F.col("longitude"),

        F.col("violation_issue_date").alias("event_date"),

        F.lit("DOB").alias("source")
    )
)


building_observations = (
    hpd_buildings
    .unionByName(dob_buildings)
)

print(
    "Building observations:",
    building_observations.count()
)

Building observations: 1075193


In [16]:
from pyspark.sql.window import Window


latest_window = (
    Window
    .partitionBy("bin")
    .orderBy(
        F.col("event_date").desc_nulls_last(),
        F.col("source").asc(),
        F.col("address").asc()
    )
)


latest_building = (
    building_observations

    .withColumn(
        "row_num",
        F.row_number().over(latest_window)
    )

    .filter(
        F.col("row_num") == 1
    )

    .drop(
        "row_num"
    )
)

In [17]:
building_history = (
    building_observations

    .groupBy("bin")

    .agg(

        F.collect_set(
            "bbl"
        ).alias(
            "bbl_aliases"
        ),

        F.collect_set(
            "address"
        ).alias(
            "address_aliases"
        ),

        F.collect_set(
            "source"
        ).alias(
            "sources"
        ),

        F.countDistinct(
            "bbl"
        ).alias(
            "bbl_count"
        ),

        F.countDistinct(
            "address"
        ).alias(
            "address_count"
        )
    )
)

In [18]:
building_master = (
    latest_building

    .join(
        building_history,
        on="bin",
        how="left"
    )

    .withColumn(
        "building_key",
        F.concat(
            F.lit("BIN:"),
            F.col("bin")
        )
    )

    .withColumn(
        "identity_status",

        F.when(
            F.col("bbl_count") > 1,
            F.lit("BBL_HISTORY")
        )

        .when(
            F.col("address_count") > 1,
            F.lit("ADDRESS_ALIAS")
        )

        .otherwise(
            F.lit("STABLE")
        )
    )
)

In [19]:
print(
    "Canonical buildings:",
    building_master.count()
)

building_master.groupBy(
    "identity_status"
).count().orderBy(
    F.desc("count")
).show(
    truncate=False
)

Canonical buildings: 197958


+---------------+------+
|identity_status|count |
+---------------+------+
|STABLE         |192224|
|ADDRESS_ALIAS  |5567  |
|BBL_HISTORY    |167   |
+---------------+------+



In [20]:
# ==================================================
# PLUTO PROPERTY REFERENCE
# ==================================================

pluto_property = (
    pluto_df

    .select(
        "bbl",

        F.col("address").alias(
            "pluto_address"
        ),

        F.col("borough").alias(
            "pluto_borough"
        ),

        "landuse",
        "bldgclass",
        "yearbuilt",
        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",
        "lotarea",
        "bldgarea",

        F.col("latitude").alias(
            "pluto_latitude"
        ),

        F.col("longitude").alias(
            "pluto_longitude"
        ),

        "snapshot_version"
    )

    .dropDuplicates(
        ["bbl"]
    )
)

In [21]:
building_with_property = (
    building_master

    .join(
        pluto_property,
        on="bbl",
        how="left"
    )

    .withColumn(
        "pluto_match_status",

        F.when(
            F.col("snapshot_version").isNotNull(),
            F.lit("MATCHED")
        )

        .otherwise(
            F.lit("UNMATCHED")
        )
    )
)

In [22]:
print(
    "Total buildings:",
    building_with_property.count()
)

building_with_property.groupBy(
    "pluto_match_status"
).count().show(
    truncate=False
)

Total buildings: 197958


+------------------+------+
|pluto_match_status|count |
+------------------+------+
|UNMATCHED         |712   |
|MATCHED           |197246|
+------------------+------+



In [23]:
matched_buildings = (
    building_with_property
    .filter(
        F.col("pluto_match_status") == "MATCHED"
    )
    .count()
)

total_buildings = (
    building_with_property.count()
)

print(
    "PLUTO building coverage:",
    f"{matched_buildings / total_buildings * 100:.2f}%"
)

PLUTO building coverage: 99.64%


In [24]:
building_with_property.filter(
    F.col("pluto_match_status") == "UNMATCHED"
).groupBy(
    "identity_status"
).count().show(
    truncate=False
)

+---------------+-----+
|identity_status|count|
+---------------+-----+
|STABLE         |643  |
|ADDRESS_ALIAS  |17   |
|BBL_HISTORY    |52   |
+---------------+-----+



In [25]:
unmatched_buildings = (
    building_with_property
    .filter(
        F.col("pluto_match_status") == "UNMATCHED"
    )
    .select(
        "bin",
        "building_key",
        "bbl",
        "bbl_aliases",
        "identity_status"
    )
)


alias_candidates = (
    unmatched_buildings
    .withColumn(
        "candidate_bbl",
        F.explode_outer("bbl_aliases")
    )
)


alias_matches = (
    alias_candidates
    .join(
        pluto_bbl_df.withColumnRenamed(
            "bbl",
            "candidate_bbl"
        ),
        on="candidate_bbl",
        how="inner"
    )
)


print(
    "Buildings recovered through BBL history:",
    alias_matches
    .select("bin")
    .distinct()
    .count()
)

Buildings recovered through BBL history: 51


In [26]:
# ==================================================
# 1. REMOVE BUILDINGS RECOVERED BY BBL HISTORY
# ==================================================

recovered_alias_bins = (
    alias_matches
    .select("bin")
    .distinct()
)


unresolved_after_alias = (
    building_with_property

    .filter(
        F.col("pluto_match_status") == "UNMATCHED"
    )

    .join(
        recovered_alias_bins,
        on="bin",
        how="left_anti"
    )
)


print(
    "Unresolved after BBL history:",
    unresolved_after_alias.count()
)

26/09/05 15:57:26 WARN MemoryStore: Not enough space to cache broadcast_259 in memory! (computed 111.0 MiB so far)


Unresolved after BBL history: 661


In [27]:
# ==================================================
# 2. ADDRESS NORMALIZATION
# ==================================================

def normalize_address(column):

    return F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.upper(column),
                r"[^A-Z0-9 ]",
                " "
            ),
            r"\s+",
            " "
        )
    )

In [28]:
# ==================================================
# 3. NORMALIZE UNRESOLVED BUILDING ADDRESSES
# ==================================================

unresolved_address_candidates = (
    unresolved_after_alias

    .withColumn(
        "normalized_address",
        normalize_address(
            F.col("address")
        )
    )

    .withColumn(
        "normalized_borough",
        F.upper(
            F.trim(
                F.col("borough")
            )
        )
    )
)

In [29]:
# ==================================================
# 4. CREATE PLUTO ADDRESS REFERENCE
# ==================================================

pluto_address_reference = (
    pluto_df

    .select(
        F.col("bbl").alias(
            "pluto_bbl_candidate"
        ),

        F.upper(
            F.trim(
                F.col("borough")
            )
        ).alias(
            "normalized_borough"
        ),

        normalize_address(
            F.col("address")
        ).alias(
            "normalized_address"
        )
    )

    .filter(
        F.col("normalized_address").isNotNull()
        & (F.col("normalized_address") != "")
    )
)

In [30]:
# ==================================================
# 5. CHECK ADDRESS UNIQUENESS IN PLUTO
# ==================================================

pluto_address_quality = (
    pluto_address_reference

    .groupBy(
        "normalized_borough",
        "normalized_address"
    )

    .agg(
        F.countDistinct(
            "pluto_bbl_candidate"
        ).alias(
            "bbl_count"
        ),

        F.collect_set(
            "pluto_bbl_candidate"
        ).alias(
            "bbl_candidates"
        )
    )
)


print(
    "Unique PLUTO address keys:",
    pluto_address_quality
    .filter(
        F.col("bbl_count") == 1
    )
    .count()
)

print(
    "Ambiguous PLUTO address keys:",
    pluto_address_quality
    .filter(
        F.col("bbl_count") > 1
    )
    .count()
)

Unique PLUTO address keys: 829295


Ambiguous PLUTO address keys: 4687


In [31]:
# ==================================================
# 6. KEEP ONLY UNIQUE PLUTO ADDRESSES
# ==================================================

unique_pluto_addresses = (
    pluto_address_quality

    .filter(
        F.col("bbl_count") == 1
    )

    .select(
        "normalized_borough",
        "normalized_address",

        F.element_at(
            F.col("bbl_candidates"),
            1
        ).alias(
            "matched_pluto_bbl"
        )
    )
)

In [32]:
# ==================================================
# 7. EXACT ADDRESS MATCH
# ==================================================

address_matches = (
    unresolved_address_candidates

    .join(
        unique_pluto_addresses,

        on=[
            "normalized_borough",
            "normalized_address"
        ],

        how="inner"
    )
)


print(
    "Buildings recovered by exact address:",
    address_matches
    .select("bin")
    .distinct()
    .count()
)

Buildings recovered by exact address: 150


In [33]:
# ==================================================
# 8. REMAINING UNRESOLVED
# ==================================================

address_recovered_bins = (
    address_matches
    .select("bin")
    .distinct()
)


still_unresolved = (
    unresolved_after_alias

    .join(
        address_recovered_bins,
        on="bin",
        how="left_anti"
    )
)


print(
    "Still unresolved:",
    still_unresolved.count()
)

26/09/05 15:59:56 WARN MemoryStore: Not enough space to cache broadcast_324 in memory! (computed 111.0 MiB so far)
26/09/05 15:59:57 WARN MemoryStore: Not enough space to cache broadcast_325 in memory! (computed 98.0 MiB so far)
26/09/05 15:59:57 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:58 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:58 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:58 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:58 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:58 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:59 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 15:59:59 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try ag

KeyboardInterrupt: 

26/09/05 16:00:26 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:26 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:26 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:27 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:27 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:27 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:27 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:27 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:28 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:28 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.
26/09/05 16:00:28 WARN TaskMem

In [1]:
import os
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from pyspark.sql import SparkSession
from minio_config import configure_minio, minio_path


spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold Identity")
    .master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/05 16:05:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.4.0
Master: local[4]
Spark UI: http://ff7529cf1dd6:4040


In [2]:
print(
    "Driver memory:",
    spark.sparkContext.getConf().get(
        "spark.driver.memory"
    )
)

print(
    "Shuffle partitions:",
    spark.conf.get(
        "spark.sql.shuffle.partitions"
    )
)

Driver memory: 4g
Shuffle partitions: 32


In [3]:
max_memory_gb = (
    spark.sparkContext
    ._jvm
    .java.lang.Runtime
    .getRuntime()
    .maxMemory()
    / 1024
    / 1024
    / 1024
)

print(
    "Actual JVM max memory:",
    round(max_memory_gb, 2),
    "GB"
)

Actual JVM max memory: 4.0 GB


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ==================================================
# LOAD SILVER
# ==================================================

pluto_df = spark.read.parquet(
    minio_path("silver/pluto/version=26v2")
)

hpd_df = spark.read.parquet(
    minio_path("silver/hpd")
)

dob_df = spark.read.parquet(
    minio_path("silver/dob")
)


# ==================================================
# NORMALIZE ADDRESS
# ==================================================

def normalize_address(column):

    return F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.upper(column),
                r"[^A-Z0-9 ]",
                " "
            ),
            r"\s+",
            " "
        )
    )


# ==================================================
# HPD BUILDING OBSERVATIONS
#
# Aggregate first to reduce the amount of data
# before the expensive Window operation.
# ==================================================

hpd_buildings = (
    hpd_df
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
    .select(
        F.trim("bin").alias("bin"),
        "bbl",

        normalize_address(
            F.concat_ws(
                " ",
                F.col("housenumber"),
                F.col("streetname")
            )
        ).alias("address"),

        F.col("boro").alias("borough"),
        "latitude",
        "longitude",
        F.col("inspectiondate").alias("event_date")
    )
    .groupBy(
        "bin",
        "bbl",
        "address",
        "borough",
        "latitude",
        "longitude"
    )
    .agg(
        F.max("event_date").alias("event_date")
    )
    .withColumn(
        "source",
        F.lit("HPD")
    )
)


# ==================================================
# DOB BUILDING OBSERVATIONS
# ==================================================

dob_buildings = (
    dob_df
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
    .select(
        F.trim("bin").alias("bin"),
        "bbl",

        normalize_address(
            F.concat_ws(
                " ",
                F.col("house_number"),
                F.col("street")
            )
        ).alias("address"),

        "borough",
        "latitude",
        "longitude",
        F.col("violation_issue_date").alias("event_date")
    )
    .groupBy(
        "bin",
        "bbl",
        "address",
        "borough",
        "latitude",
        "longitude"
    )
    .agg(
        F.max("event_date").alias("event_date")
    )
    .withColumn(
        "source",
        F.lit("DOB")
    )
)


building_observations = (
    hpd_buildings
    .unionByName(dob_buildings)
)


# ==================================================
# LATEST OBSERVATION PER BIN
# ==================================================

latest_window = (
    Window
    .partitionBy("bin")
    .orderBy(
        F.col("event_date").desc_nulls_last(),
        F.col("source").asc()
    )
)


latest_building = (
    building_observations
    .withColumn(
        "rn",
        F.row_number().over(latest_window)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)


# ==================================================
# BBL HISTORY
# ==================================================

building_history = (
    building_observations
    .groupBy("bin")
    .agg(
        F.collect_set("bbl").alias(
            "bbl_aliases"
        )
    )
)


building_master = (
    latest_building
    .join(
        building_history,
        on="bin",
        how="left"
    )
)


# ==================================================
# PLUTO BBL REFERENCE
# ==================================================

pluto_bbl_df = (
    pluto_df
    .select("bbl")
    .filter(
        F.col("bbl").isNotNull()
    )
    .dropDuplicates()
)


# ==================================================
# BUILDINGS WITHOUT CURRENT BBL IN PLUTO
# ==================================================

direct_unmatched = (
    building_master
    .join(
        pluto_bbl_df,
        on="bbl",
        how="left_anti"
    )
)


print(
    "Direct unmatched:",
    direct_unmatched.count()
)

26/09/05 16:08:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Direct unmatched: 712


In [5]:
alias_matches = (
    direct_unmatched

    .select(
        "bin",
        F.explode_outer(
            "bbl_aliases"
        ).alias(
            "candidate_bbl"
        )
    )

    .join(
        pluto_bbl_df
        .withColumnRenamed(
            "bbl",
            "candidate_bbl"
        ),
        on="candidate_bbl",
        how="inner"
    )

    .select("bin")
    .distinct()
)


print(
    "Recovered through BBL history:",
    alias_matches.count()
)


unresolved_after_alias = (
    direct_unmatched

    .join(
        alias_matches,
        on="bin",
        how="left_anti"
    )
)


print(
    "Unresolved after BBL history:",
    unresolved_after_alias.count()
)

Recovered through BBL history: 51


Unresolved after BBL history: 661


In [6]:
# ==================================================
# CUT THE LONG SPARK LINEAGE
# ==================================================

unresolved_after_alias = (
    unresolved_after_alias
    .localCheckpoint(eager=True)
)

print(
    "Checkpointed unresolved buildings:",
    unresolved_after_alias.count()
)

Checkpointed unresolved buildings: 661


In [7]:
# ==================================================
# 1. PREPARE THE 661 BUILDINGS
# ==================================================

unresolved_address_keys = (
    unresolved_after_alias

    .select(
        "bin",
        "address",
        "borough",
        "latitude",
        "longitude"
    )

    .withColumn(
        "normalized_address",
        normalize_address(
            F.col("address")
        )
    )

    .withColumn(
        "normalized_borough",
        F.upper(
            F.trim(
                F.col("borough")
            )
        )
    )

    .filter(
        F.col("normalized_address").isNotNull()
        & (F.col("normalized_address") != "")
    )
)


print(
    "Buildings with usable address:",
    unresolved_address_keys.count()
)

Buildings with usable address: 578


In [8]:
# ==================================================
# 2. MINIMAL PLUTO ADDRESS REFERENCE
# ==================================================

pluto_address_reference = (
    pluto_df

    .select(
        F.col("bbl").alias(
            "matched_pluto_bbl"
        ),

        F.upper(
            F.trim(
                F.col("borough")
            )
        ).alias(
            "normalized_borough"
        ),

        normalize_address(
            F.col("address")
        ).alias(
            "normalized_address"
        )
    )

    .filter(
        F.col("normalized_address").isNotNull()
        & (F.col("normalized_address") != "")
    )
)

In [9]:
# ==================================================
# 3. EXACT ADDRESS MATCH
#
# The 661 buildings are broadcast because
# they are tiny compared with PLUTO.
# ==================================================

address_candidates = (
    pluto_address_reference

    .join(
        F.broadcast(
            unresolved_address_keys
        ),

        on=[
            "normalized_borough",
            "normalized_address"
        ],

        how="inner"
    )
)

In [10]:
# ==================================================
# 4. CHECK MATCH AMBIGUITY
# ==================================================

address_match_quality = (
    address_candidates

    .groupBy("bin")

    .agg(
        F.countDistinct(
            "matched_pluto_bbl"
        ).alias(
            "pluto_bbl_count"
        ),

        F.first(
            "matched_pluto_bbl"
        ).alias(
            "matched_pluto_bbl"
        )
    )
)


exact_address_matches = (
    address_match_quality

    .filter(
        F.col("pluto_bbl_count") == 1
    )
)


ambiguous_address_matches = (
    address_match_quality

    .filter(
        F.col("pluto_bbl_count") > 1
    )
)


print(
    "Recovered by exact address:",
    exact_address_matches.count()
)

print(
    "Ambiguous address matches:",
    ambiguous_address_matches.count()
)

Recovered by exact address: 150


Ambiguous address matches: 12


In [11]:
# ==================================================
# 5. STILL UNRESOLVED
# ==================================================

still_unresolved = (
    unresolved_after_alias

    .join(
        exact_address_matches
        .select("bin"),

        on="bin",
        how="left_anti"
    )
)


print(
    "Still unresolved:",
    still_unresolved.count()
)

Still unresolved: 511


In [12]:
git status


SyntaxError: invalid syntax (339777499.py, line 1)

In [13]:
spark.stop()

In [14]:
[
    name
    for name, obj in globals().items()
    if obj.__class__.__name__ == "DataFrame"
    and "unresolved" in name.lower()
]

['unresolved_after_alias', 'unresolved_address_keys', 'still_unresolved']

In [15]:
print("still_unresolved count:", still_unresolved.count())

still_unresolved.printSchema()

Py4JJavaError: An error occurred while calling o341.count.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.wa

In [16]:
spark.range(1).count()

AttributeError: 'NoneType' object has no attribute 'sc'

In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from minio_config import configure_minio


spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold Identity")
    .master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/05 19:04:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.4.0
Master: local[4]
Test: 1


In [2]:
from minio_config import minio_path

pluto_df = spark.read.parquet(
    minio_path("silver/pluto/version=26v2")
)

hpd_df = spark.read.parquet(
    minio_path("silver/hpd")
)

dob_df = spark.read.parquet(
    minio_path("silver/dob")
)

nyc311_df = spark.read.parquet(
    minio_path("silver/nyc_311")
)

print("=== HPD ===")
hpd_df.printSchema()

print("\n=== DOB ===")
dob_df.printSchema()

print("\n=== PLUTO ===")
pluto_df.printSchema()

26/09/05 19:04:48 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


=== HPD ===
root
 |-- violationid: string (nullable = true)
 |-- novid: string (nullable = true)
 |-- buildingid: string (nullable = true)
 |-- registrationid: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- boroid: string (nullable = true)
 |-- boro: string (nullable = true)
 |-- block: string (nullable = true)
 |-- lot: string (nullable = true)
 |-- housenumber: string (nullable = true)
 |-- lowhousenumber: string (nullable = true)
 |-- highhousenumber: string (nullable = true)
 |-- streetname: string (nullable = true)
 |-- apartment: string (nullable = true)
 |-- story: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- class: string (nullable = true)
 |-- violationstatus: string (nullable = true)
 |-- currentstatus: string (nullable = true)
 |-- currentstatusid: string (nullable = true)
 |-- inspectiondate: timestamp (nullable = true)
 |-- novissueddate: timestamp (nullable = true)
 |-- currentstatusdate: timestamp (

In [3]:
from pyspark.sql import functions as F


def normalize_borough(col):
    value = F.upper(F.trim(col))

    return (
        F.when(value.isin("1", "MN", "MANHATTAN"), "MANHATTAN")
         .when(value.isin("2", "BX", "BRONX"), "BRONX")
         .when(value.isin("3", "BK", "BROOKLYN"), "BROOKLYN")
         .when(value.isin("4", "QN", "QUEENS"), "QUEENS")
         .when(
             value.isin(
                 "5",
                 "SI",
                 "STATEN ISLAND"
             ),
             "STATEN ISLAND"
         )
         .otherwise(value)
    )


# -----------------------------
# HPD observations
# -----------------------------

hpd_obs = (
    hpd_df
    .filter(F.col("bin").isNotNull())
    .select(
        F.col("bin"),
        F.col("bbl"),

        F.trim(
            F.concat_ws(
                " ",
                F.col("housenumber"),
                F.col("streetname")
            )
        ).alias("address"),

        normalize_borough(
            F.col("boro")
        ).alias("borough"),

        F.col("latitude"),
        F.col("longitude"),

        F.col(
            "inspectiondate"
        ).alias("event_date"),

        F.lit("HPD").alias("source")
    )
)


# -----------------------------
# DOB observations
# -----------------------------

dob_obs = (
    dob_df
    .filter(F.col("bin").isNotNull())
    .select(
        F.col("bin"),
        F.col("bbl"),

        F.trim(
            F.concat_ws(
                " ",
                F.col("house_number"),
                F.col("street")
            )
        ).alias("address"),

        normalize_borough(
            F.col("borough")
        ).alias("borough"),

        F.col("latitude"),
        F.col("longitude"),

        F.col(
            "violation_issue_date"
        ).alias("event_date"),

        F.lit("DOB").alias("source")
    )
)


# -----------------------------
# Combine HPD + DOB
# -----------------------------

building_observations = (
    hpd_obs
    .unionByName(dob_obs)
)


OBS_PATH = minio_path(
    "gold/building_identity/intermediate/building_observations"
)


(
    building_observations
    .write
    .mode("overwrite")
    .parquet(OBS_PATH)
)

print("Building observations saved successfully")

Building observations saved successfully


In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ==================================================
# READ SAVED BUILDING OBSERVATIONS
# ==================================================

OBS_PATH = minio_path(
    "gold/building_identity/intermediate/building_observations"
)

obs_df = (
    spark.read
    .parquet(OBS_PATH)
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
)


# ==================================================
# LATEST OBSERVATION PER BIN
# ==================================================

latest_window = (
    Window
    .partitionBy("bin")
    .orderBy(
        F.col("event_date").desc_nulls_last(),
        F.col("source").asc(),
        F.col("bbl").asc_nulls_last(),
        F.col("address").asc_nulls_last()
    )
)


latest_building = (
    obs_df
    .withColumn(
        "rn",
        F.row_number().over(latest_window)
    )
    .filter(
        F.col("rn") == 1
    )
    .select(
        "bin",
        F.col("bbl").alias("current_bbl"),
        F.col("address").alias("current_address"),
        F.col("borough"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("event_date").alias("latest_event_date"),
        F.col("source").alias("latest_source")
    )
)


# ==================================================
# BUILD HISTORY PER BIN
# ==================================================

building_history = (
    obs_df
    .groupBy("bin")
    .agg(
        F.collect_set("bbl").alias("bbl_aliases"),
        F.collect_set("address").alias("address_aliases"),
        F.collect_set("source").alias("sources"),

        F.countDistinct("bbl").alias("bbl_count"),
        F.countDistinct("address").alias("address_count")
    )
)


# ==================================================
# CANONICAL BUILDING MASTER
# ==================================================

building_master = (
    latest_building
    .join(
        building_history,
        on="bin",
        how="left"
    )

    .withColumn(
        "building_key",
        F.concat(
            F.lit("BIN:"),
            F.col("bin")
        )
    )

    .withColumn(
        "identity_status",
        F.when(
            F.col("bbl_count") > 1,
            F.lit("BBL_HISTORY")
        )
        .when(
            F.col("address_count") > 1,
            F.lit("ADDRESS_ALIAS")
        )
        .otherwise(
            F.lit("STABLE")
        )
    )
)


# ==================================================
# SAVE BUILDING MASTER
# ==================================================

BUILDING_MASTER_PATH = minio_path(
    "gold/building_identity/building_master"
)

(
    building_master
    .write
    .mode("overwrite")
    .parquet(BUILDING_MASTER_PATH)
)

print("Building Master saved successfully")

Building Master saved successfully


In [5]:
BUILDING_MASTER_PATH = minio_path(
    "gold/building_identity/building_master"
)

building_master_df = (
    spark.read
    .parquet(BUILDING_MASTER_PATH)
)

print(
    "Total buildings:",
    building_master_df.count()
)

building_master_df \
    .groupBy("identity_status") \
    .count() \
    .orderBy("identity_status") \
    .show(truncate=False)

Total buildings: 197958
+---------------+------+
|identity_status|count |
+---------------+------+
|ADDRESS_ALIAS  |5567  |
|BBL_HISTORY    |167   |
|STABLE         |192224|
+---------------+------+



In [6]:
from pyspark.sql import functions as F


# ==================================================
# MINIMAL PLUTO REFERENCE
# ==================================================

pluto_ref = (
    pluto_df
    .select(
        F.col("bbl").alias("pluto_bbl"),
        F.col("address").alias("pluto_address"),
        F.col("borough").alias("pluto_borough"),
        F.col("latitude").alias("pluto_latitude"),
        F.col("longitude").alias("pluto_longitude"),
        F.col("yearbuilt"),
        F.col("landuse"),
        F.col("bldgclass")
    )
)


# ==================================================
# DIRECT BBL MATCH
# ==================================================

direct_join = (
    building_master_df
    .join(
        pluto_ref,
        building_master_df["current_bbl"] == pluto_ref["pluto_bbl"],
        how="left"
    )
)


# ==================================================
# KEEP ONLY UNMATCHED BUILDINGS
# ==================================================

direct_unmatched = (
    direct_join
    .filter(
        F.col("pluto_bbl").isNull()
    )
    .select(
        building_master_df["*"]
    )
)


# ==================================================
# SAVE IMMEDIATELY
# ==================================================

DIRECT_UNMATCHED_PATH = minio_path(
    "gold/building_identity/intermediate/direct_bbl_unmatched"
)

(
    direct_unmatched
    .write
    .mode("overwrite")
    .parquet(DIRECT_UNMATCHED_PATH)
)

print("Direct BBL unmatched saved successfully")

Direct BBL unmatched saved successfully


In [7]:
direct_unmatched_df = spark.read.parquet(
    DIRECT_UNMATCHED_PATH
)

print(
    "Direct BBL unmatched:",
    direct_unmatched_df.count()
)

Direct BBL unmatched: 712


In [8]:
from pyspark.sql import functions as F


# ==================================================
# EXPLODE BBL HISTORY
# ==================================================

alias_candidates = (
    direct_unmatched_df
    .select(
        "bin",
        "current_bbl",
        F.explode_outer("bbl_aliases").alias("alias_bbl")
    )
    .filter(
        F.col("alias_bbl").isNotNull()
    )
)


# ==================================================
# MATCH HISTORICAL BBL AGAINST PLUTO
# ==================================================

alias_matches_raw = (
    alias_candidates
    .join(
        pluto_ref.select("pluto_bbl"),
        F.col("alias_bbl") == F.col("pluto_bbl"),
        how="inner"
    )
)


# ==================================================
# COUNT UNIQUE PLUTO MATCHES PER BIN
# ==================================================

alias_match_summary = (
    alias_matches_raw
    .groupBy("bin")
    .agg(
        F.collect_set("pluto_bbl").alias("matched_alias_bbls"),
        F.countDistinct("pluto_bbl").alias("alias_match_count")
    )
)


# ==================================================
# ONLY SAFE MATCHES:
# exactly one historical BBL matches PLUTO
# ==================================================

alias_recovered_keys = (
    alias_match_summary
    .filter(
        F.col("alias_match_count") == 1
    )
    .select(
        "bin",
        F.element_at(
            F.col("matched_alias_bbls"),
            1
        ).alias("resolved_bbl")
    )
)


# ==================================================
# FULL RECOVERED BUILDING RECORDS
# ==================================================

alias_recovered = (
    direct_unmatched_df
    .join(
        alias_recovered_keys,
        on="bin",
        how="inner"
    )
    .withColumn(
        "match_method",
        F.lit("BBL_HISTORY")
    )
    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )
)


# ==================================================
# BUILDINGS STILL UNRESOLVED
# ==================================================

unresolved_after_alias = (
    direct_unmatched_df
    .join(
        alias_recovered_keys.select("bin"),
        on="bin",
        how="left_anti"
    )
)


# ==================================================
# SAVE BOTH DATASETS
# ==================================================

ALIAS_RECOVERED_PATH = minio_path(
    "gold/building_identity/intermediate/bbl_history_recovered"
)

UNRESOLVED_ALIAS_PATH = minio_path(
    "gold/building_identity/intermediate/unresolved_after_bbl_history"
)


(
    alias_recovered
    .write
    .mode("overwrite")
    .parquet(ALIAS_RECOVERED_PATH)
)

(
    unresolved_after_alias
    .write
    .mode("overwrite")
    .parquet(UNRESOLVED_ALIAS_PATH)
)

print("BBL history stage saved successfully")

BBL history stage saved successfully


In [9]:
alias_recovered_df = spark.read.parquet(
    ALIAS_RECOVERED_PATH
)

unresolved_after_alias_df = spark.read.parquet(
    UNRESOLVED_ALIAS_PATH
)

print(
    "Recovered by BBL history:",
    alias_recovered_df.count()
)

print(
    "Still unresolved:",
    unresolved_after_alias_df.count()
)

alias_match_summary \
    .groupBy("alias_match_count") \
    .count() \
    .orderBy("alias_match_count") \
    .show()

Recovered by BBL history: 51
Still unresolved: 661
+-----------------+-----+
|alias_match_count|count|
+-----------------+-----+
|                1|   51|
+-----------------+-----+



In [10]:
from pyspark.sql import functions as F


# ==================================================
# NORMALIZE ADDRESS
# ==================================================

def normalize_address(column):
    return F.trim(
        F.regexp_replace(
            F.regexp_replace(
                F.upper(column),
                r"[^A-Z0-9 ]",
                " "
            ),
            r"\s+",
            " "
        )
    )


# ==================================================
# BUILD ADDRESS KEYS FOR UNRESOLVED BUILDINGS
# ==================================================

unresolved_address_keys = (
    unresolved_after_alias_df

    # current_address + all historical addresses
    .withColumn(
        "all_addresses",
        F.array_distinct(
            F.concat(
                F.array(F.col("current_address")),
                F.col("address_aliases")
            )
        )
    )

    .select(
        "bin",
        "borough",
        F.explode_outer("all_addresses").alias("source_address")
    )

    .filter(
        F.col("source_address").isNotNull()
    )

    .withColumn(
        "normalized_address",
        normalize_address(
            F.col("source_address")
        )
    )

    .filter(
        F.col("normalized_address") != ""
    )

    .dropDuplicates(
        [
            "bin",
            "borough",
            "normalized_address"
        ]
    )
)


# ==================================================
# MINIMAL PLUTO ADDRESS REFERENCE
# ==================================================

pluto_address_ref = (
    pluto_df

    .select(
        F.col("bbl").alias("pluto_bbl"),
        F.col("address").alias("pluto_address"),
        F.col("borough").alias("pluto_borough")
    )

    .filter(
        F.col("pluto_bbl").isNotNull()
        & F.col("pluto_address").isNotNull()
    )

    .withColumn(
        "normalized_address",
        normalize_address(
            F.col("pluto_address")
        )
    )

    .dropDuplicates(
        [
            "pluto_bbl",
            "pluto_borough",
            "normalized_address"
        ]
    )
)


# ==================================================
# EXACT ADDRESS + BOROUGH MATCH
# ==================================================

exact_address_matches_raw = (
    unresolved_address_keys

    .join(
        F.broadcast(pluto_address_ref),

        (
            unresolved_address_keys["normalized_address"]
            ==
            pluto_address_ref["normalized_address"]
        )
        &
        (
            unresolved_address_keys["borough"]
            ==
            pluto_address_ref["pluto_borough"]
        ),

        how="inner"
    )
)


# ==================================================
# HOW MANY PLUTO BBLs MATCH EACH BIN?
# ==================================================

exact_address_summary = (
    exact_address_matches_raw

    .groupBy("bin")

    .agg(
        F.collect_set(
            "pluto_bbl"
        ).alias("matched_pluto_bbls"),

        F.countDistinct(
            "pluto_bbl"
        ).alias("address_match_count")
    )
)


# ==================================================
# SAFE EXACT MATCHES
# exactly one PLUTO BBL
# ==================================================

exact_recovered_keys = (
    exact_address_summary

    .filter(
        F.col("address_match_count") == 1
    )

    .select(
        "bin",

        F.element_at(
            F.col("matched_pluto_bbls"),
            1
        ).alias("resolved_bbl")
    )
)


# ==================================================
# AMBIGUOUS EXACT MATCHES
# more than one PLUTO BBL
# ==================================================

exact_ambiguous = (
    exact_address_summary

    .filter(
        F.col("address_match_count") > 1
    )
)


# ==================================================
# FULL RECOVERED BUILDINGS
# ==================================================

exact_address_recovered = (
    unresolved_after_alias_df

    .join(
        exact_recovered_keys,
        on="bin",
        how="inner"
    )

    .withColumn(
        "match_method",
        F.lit("EXACT_ADDRESS")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )
)


# ==================================================
# STILL UNRESOLVED
# ambiguous matches remain unresolved
# ==================================================

still_unresolved = (
    unresolved_after_alias_df

    .join(
        exact_recovered_keys.select("bin"),
        on="bin",
        how="left_anti"
    )
)


# ==================================================
# SAVE RESULTS IMMEDIATELY
# ==================================================

EXACT_RECOVERED_PATH = minio_path(
    "gold/building_identity/intermediate/exact_address_recovered"
)

EXACT_AMBIGUOUS_PATH = minio_path(
    "gold/building_identity/intermediate/exact_address_ambiguous"
)

STILL_UNRESOLVED_PATH = minio_path(
    "gold/building_identity/unresolved"
)


(
    exact_address_recovered
    .write
    .mode("overwrite")
    .parquet(EXACT_RECOVERED_PATH)
)

(
    exact_ambiguous
    .write
    .mode("overwrite")
    .parquet(EXACT_AMBIGUOUS_PATH)
)

(
    still_unresolved
    .write
    .mode("overwrite")
    .parquet(STILL_UNRESOLVED_PATH)
)

print("Exact address stage saved successfully")

Exact address stage saved successfully


In [11]:
exact_recovered_df = spark.read.parquet(
    EXACT_RECOVERED_PATH
)

exact_ambiguous_df = spark.read.parquet(
    EXACT_AMBIGUOUS_PATH
)

still_unresolved_df = spark.read.parquet(
    STILL_UNRESOLVED_PATH
)

print(
    "Recovered by exact address:",
    exact_recovered_df.count()
)

print(
    "Ambiguous exact matches:",
    exact_ambiguous_df.count()
)

print(
    "Still unresolved:",
    still_unresolved_df.count()
)

Recovered by exact address: 153
Ambiguous exact matches: 12
Still unresolved: 508


In [12]:
still_unresolved_df.printSchema()

root
 |-- bin: string (nullable = true)
 |-- current_bbl: string (nullable = true)
 |-- current_address: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latest_event_date: timestamp (nullable = true)
 |-- latest_source: string (nullable = true)
 |-- bbl_aliases: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- address_aliases: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- sources: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- bbl_count: long (nullable = true)
 |-- address_count: long (nullable = true)
 |-- building_key: string (nullable = true)
 |-- identity_status: string (nullable = true)



In [13]:
still_unresolved_df.select(
    "bin",
    "current_bbl",
    "current_address",
    "borough",
    "latitude",
    "longitude"
).show(10, truncate=False)

+-------+-----------+--------------------+---------+---------+----------+
|bin    |current_bbl|current_address     |borough  |latitude |longitude |
+-------+-----------+--------------------+---------+---------+----------+
|1033212|1012350033 |568 AMSTERDAM AVENUE|MANHATTAN|40.788677|-73.974302|
|1060006|null       |                    |null     |null     |null      |
|2115800|2023440011 |275 GRAND CONCOURSE |BRONX    |40.814118|-73.929586|
|3013684|3008010032 |664 51 STREET       |BROOKLYN |40.642403|-74.007369|
|3015852|3008490040 |5722 7 AVENUE       |BROOKLYN |40.638357|-74.010179|
|3049594|3017780004 |983 BEDFORD AVENUE  |BROOKLYN |40.690458|-73.955312|
|3070664|3030030027 |47 GRATTAN STREET   |BROOKLYN |40.705751|-73.931169|
|3072477|3031840025 |128 TROUTMAN STREET |BROOKLYN |40.69975 |-73.929552|
|3136330|3056080001 |1001 45 STREET      |BROOKLYN |40.641116|-73.996148|
|3136664|3056150028 |1140 45 STREET      |BROOKLYN |40.639364|-73.993316|
+-------+-----------+-----------------

In [14]:
spark.stop()

In [15]:
from pyspark.sql import functions as F
from minio_config import minio_path


# ==================================================
# READ PHYSICAL GOLD DATASETS
# ==================================================

building_master_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/building_master"
    )
)

alias_recovered_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/intermediate/bbl_history_recovered"
    )
)

exact_recovered_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/intermediate/exact_address_recovered"
    )
)

resolver_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/elasticsearch_resolver"
    )
)

pluto_df = spark.read.parquet(
    minio_path(
        "silver/pluto/version=26v2"
    )
)

Py4JJavaError: An error occurred while calling o547.parquet.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.

In [16]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from minio_config import configure_minio, minio_path


# ==================================================
# NEW SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Final Gold")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")


print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Spark version: 3.4.0
Master: local[2]
Test: 1


In [17]:
building_master_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/building_master"
    )
)

alias_recovered_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/intermediate/bbl_history_recovered"
    )
)

exact_recovered_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/intermediate/exact_address_recovered"
    )
)

resolver_df = spark.read.parquet(
    minio_path(
        "gold/building_identity/elasticsearch_resolver"
    )
)

pluto_df = spark.read.parquet(
    minio_path(
        "silver/pluto/version=26v2"
    )
)

print("Gold sources loaded successfully")

Gold sources loaded successfully


In [18]:
from pyspark.sql import functions as F


# ==================================================
# 1. DIRECT BBL MATCH
# ==================================================

pluto_bbl_ref = (
    pluto_df
    .select(
        F.col("bbl").alias("pluto_bbl")
    )
    .dropDuplicates()
)


direct_resolved = (
    building_master_df

    .join(
        pluto_bbl_ref,
        building_master_df["current_bbl"]
        ==
        pluto_bbl_ref["pluto_bbl"],
        "inner"
    )

    .select(
        "bin",
        F.col("current_bbl").alias("resolved_bbl")
    )

    .withColumn(
        "match_method",
        F.lit("DIRECT_BBL")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# 2. BBL HISTORY MATCH
# ==================================================

alias_resolved = (
    alias_recovered_df

    .select(
        "bin",
        "resolved_bbl"
    )

    .withColumn(
        "match_method",
        F.lit("BBL_HISTORY")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# 3. EXACT ADDRESS MATCH
# ==================================================

exact_resolved = (
    exact_recovered_df

    .select(
        "bin",
        "resolved_bbl"
    )

    .withColumn(
        "match_method",
        F.lit("EXACT_ADDRESS")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# 4. ELASTICSEARCH HIGH ONLY
# ==================================================

elastic_resolved = (
    resolver_df

    .filter(
        F.col("resolution_status")
        ==
        "AUTO_RESOLVED"
    )

    .select(
        "bin",
        F.col("candidate_bbl").alias("resolved_bbl")
    )

    .withColumn(
        "match_method",
        F.lit("ELASTICSEARCH")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("RESOLVED")
    )
)


# ==================================================
# COMBINE ALL RESOLVED BUILDINGS
# ==================================================

resolved_map = (
    direct_resolved
    .unionByName(alias_resolved)
    .unionByName(exact_resolved)
    .unionByName(elastic_resolved)
)


print("Resolved rows:", resolved_map.count())

print(
    "Distinct BIN:",
    resolved_map
    .select("bin")
    .distinct()
    .count()
)

resolved_map \
    .groupBy("match_method") \
    .count() \
    .orderBy("match_method") \
    .show(truncate=False)

Resolved rows: 197459


Distinct BIN: 197459


+-------------+------+
|match_method |count |
+-------------+------+
|BBL_HISTORY  |51    |
|DIRECT_BBL   |197246|
|ELASTICSEARCH|9     |
|EXACT_ADDRESS|153   |
+-------------+------+



In [19]:
from pyspark.sql import functions as F


# ==================================================
# BUILD FINAL IDENTITY TABLE
# ==================================================

building_identity_final = (
    building_master_df

    .join(
        resolved_map,
        on="bin",
        how="left"
    )

    # Buildings that were not resolved
    .withColumn(
        "resolution_status",
        F.coalesce(
            F.col("resolution_status"),
            F.lit("UNRESOLVED")
        )
    )

    .withColumn(
        "match_method",
        F.coalesce(
            F.col("match_method"),
            F.lit("UNRESOLVED")
        )
    )

    .withColumn(
        "match_confidence",
        F.coalesce(
            F.col("match_confidence"),
            F.lit("NONE")
        )
    )
)


# ==================================================
# PLUTO PROPERTY ATTRIBUTES
# ==================================================

pluto_gold_ref = (
    pluto_df

    .select(
        F.col("bbl").alias("resolved_bbl"),

        F.col("address").alias("pluto_address"),
        F.col("borough").alias("pluto_borough"),

        F.col("latitude").alias("pluto_latitude"),
        F.col("longitude").alias("pluto_longitude"),

        "yearbuilt",
        "landuse",
        "bldgclass",
        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",
        "lotarea",
        "bldgarea",
        "ownername",
        "ownertype",
        "zipcode",
        "snapshot_version"
    )

    .dropDuplicates(
        ["resolved_bbl"]
    )
)


# ==================================================
# BUILD GOLD
# ==================================================

building_identity_gold = (
    building_identity_final

    .join(
        pluto_gold_ref,
        on="resolved_bbl",
        how="left"
    )
)


# ==================================================
# SAVE FINAL GOLD
# ==================================================

FINAL_GOLD_PATH = minio_path(
    "gold/building_identity/final"
)

(
    building_identity_gold
    .write
    .mode("overwrite")
    .parquet(FINAL_GOLD_PATH)
)

print(
    "Final Building Identity Gold saved successfully"
)

print(
    "Path:",
    FINAL_GOLD_PATH
)

26/09/05 19:26:40 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Final Building Identity Gold saved successfully
Path: s3a://nyc-building-risk/gold/building_identity/final


In [20]:
final_gold_df = spark.read.parquet(
    FINAL_GOLD_PATH
)


# ==================================================
# BASIC VALIDATION
# ==================================================

total_buildings = final_gold_df.count()

distinct_bins = (
    final_gold_df
    .select("bin")
    .distinct()
    .count()
)

resolved_buildings = (
    final_gold_df
    .filter(
        F.col("resolution_status") == "RESOLVED"
    )
    .count()
)

unresolved_buildings = (
    final_gold_df
    .filter(
        F.col("resolution_status") == "UNRESOLVED"
    )
    .count()
)


print("Total buildings:", total_buildings)
print("Distinct BIN:", distinct_bins)
print("Resolved:", resolved_buildings)
print("Unresolved:", unresolved_buildings)

print(
    "Resolution coverage:",
    round(
        resolved_buildings
        / total_buildings
        * 100,
        2
    ),
    "%"
)


# ==================================================
# RESOLUTION BREAKDOWN
# ==================================================

(
    final_gold_df

    .groupBy(
        "resolution_status",
        "match_method"
    )

    .count()

    .orderBy(
        "resolution_status",
        "match_method"
    )

    .show(
        100,
        truncate=False
    )
)

Total buildings: 197958
Distinct BIN: 197958
Resolved: 197459
Unresolved: 499
Resolution coverage: 99.75 %
+-----------------+-------------+------+
|resolution_status|match_method |count |
+-----------------+-------------+------+
|RESOLVED         |BBL_HISTORY  |51    |
|RESOLVED         |DIRECT_BBL   |197246|
|RESOLVED         |ELASTICSEARCH|9     |
|RESOLVED         |EXACT_ADDRESS|153   |
|UNRESOLVED       |UNRESOLVED   |499   |
+-----------------+-------------+------+

